# Notebook 10 — Optimization and the Training Loop

    ## Learning objectives

    - Implement forward, backward, clipping, optimizer, and scheduler steps
- Explain AdamW, warmup, weight decay, and gradient norms
- Recognize underfitting, overfitting, divergence, and data bugs

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 10.1 One optimizer update

A robust update is: fetch batch → forward → masked mean loss → backward → optionally
unscale → clip → optimizer step → scheduler step → zero gradients. AdamW maintains two
optimizer states per trained parameter, so optimizer memory can exceed weight memory.
Warmup limits unstable early updates; decay schedules reduce step size later.


In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(16, 64), nn.GELU(), nn.Linear(64, 8))
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=10)

for step in range(30):
    x = torch.randn(32, 16)
    target = torch.randint(0, 8, (32,))
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.cross_entropy(model(x), target)
    loss.backward()
    grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step(); scheduler.step()
    if step % 5 == 0:
        print(step, f"loss={loss.item():.3f}", f"grad_norm={grad_norm:.3f}",
              f"lr={scheduler.get_last_lr()[0]:.2e}")


## 10.2 Debug the learning dynamics

Log train and validation loss, learning rate, gradient norm, tokens/second, memory,
skipped/overflowed steps, and data samples. A flat loss may mean bad labels or tiny LR;
NaNs may mean overflow, invalid data, or excessive LR; falling train loss with rising
validation loss indicates overfitting or distribution mismatch. Resume tests must prove
that model, optimizer, scheduler, RNG, and dataloader state restore correctly.


## 10.3 AdamW under the hood

Adam tracks exponentially decayed first and second gradient moments:
\(m_t=\beta_1m_{t-1}+(1-\beta_1)g_t\) and
\(v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2\). Bias correction compensates for zero
initialization; the update divides by \(\sqrt{\hat v_t}+\epsilon\). AdamW applies weight
decay separately from that adaptive gradient update. Biases and normalization scale
parameters are often excluded from decay.

The learning rate is the most consequential hyperparameter, but its safe range depends on
model scale, batch/tokens per update, optimizer, precision, data, adapter/full tuning, and
initialization. Warmup prevents large early adaptive updates while moments are unreliable.
Linear or cosine decay is common; constant schedules can work for short fine-tuning. Specify
warmup in steps or token proportion and log the actual schedule.


In [ ]:
# Visualize common warmup/decay schedules independent of Trainer.
import math, matplotlib.pyplot as plt
total, warmup = 1000, 100
def multiplier(step, kind):
    if step < warmup: return step / max(warmup, 1)
    progress = (step - warmup) / (total - warmup)
    if kind == "linear": return 1 - progress
    if kind == "cosine": return .5 * (1 + math.cos(math.pi * progress))
    return 1.0
steps = range(total)
for kind in ["constant", "linear", "cosine"]:
    plt.plot(steps, [multiplier(s, kind) for s in steps], label=kind)
plt.xlabel("optimizer step"); plt.ylabel("LR multiplier"); plt.legend(); plt.show()


## 10.4 A production-quality training loop

Set model mode deliberately: `train()` enables dropout; `eval()` disables it. Move batches
to the correct device, use autocast for selected precision, divide loss for accumulation,
backward, unscale before gradient clipping, update only at accumulation boundaries, advance
the scheduler per optimizer update, and zero gradients with `set_to_none=True`. Handle a
final partial accumulation window correctly. Evaluation runs under inference/no-grad mode
and restores training mode afterward.

Distributed training adds sampler epochs, synchronized metrics, and main-process-only writes.
Reproducibility needs Python/NumPy/framework seeds, data sampler state, deterministic choices
where available, and environment capture. Exact bitwise reproducibility may still fail across
hardware and kernels; define the level you require. Checkpoints need model/adapters, optimizer,
scheduler, scaler, RNG, step/epoch, data position, and config. Test resume, do not merely save.


In [ ]:
# A reusable evaluation function illustrating reduction by examples.
@torch.inference_mode()
def evaluate_classifier(model, batches):
    was_training = model.training
    model.eval()
    total_loss = total_correct = total_items = 0
    for x, target in batches:
        logits = model(x)
        total_loss += torch.nn.functional.cross_entropy(
            logits, target, reduction="sum").item()
        total_correct += (logits.argmax(-1) == target).sum().item()
        total_items += target.numel()
    model.train(was_training)
    return {"loss": total_loss / total_items,
            "accuracy": total_correct / total_items}

validation = [(torch.randn(8, 16), torch.randint(0, 8, (8,))) for _ in range(3)]
print(evaluate_classifier(model, validation))


## 10.5 Reading training curves and running ablations

Compare train and validation loss on identical reduction/token policies. A sudden loss spike
may be a rare long batch, bad record, overflow, schedule discontinuity, or distributed issue;
log sample IDs and tokens/update. Smooth curves for visualization but retain raw values.
Validation loss can improve while task behavior regresses, so run task evals at checkpoints.
Inspect qualitative generations under fixed decoding.

Change one factor per ablation when possible: LR, warmup, effective batch, max length, masking,
LoRA rank/targets, data mixture. Report compute/tokens, not only epochs. Compare against a
no-training baseline and a simple prompting/RAG baseline. Stop based on held-out evidence and
budget, not because a predetermined epoch count completed. Archive failed runs: knowing that
a configuration diverged is useful evidence when metadata is complete.

**Minimum run record:** code/data/model revisions, tokenizer/template, seed, precision,
hardware/world size, optimizer/schedule, effective tokens/update, step/tokens seen, gradient
statistics, checkpoint IDs, eval results, and wall-clock/cost.


## 10.6 Optimization reference

| Symptom | First checks |
|---|---|
| Loss unchanged | Labels/mask, trainable params, LR, optimizer step, detached graph |
| Immediate NaN/Inf | Input values, LR, precision/scale, invalid targets, normalization |
| Periodic spikes | Batch length/source, accumulation boundary, scheduler, bad records |
| Train improves, validation worsens | Leakage-free split, overfit, distribution, regularization |
| Resume diverges | RNG, sampler/data position, optimizer/scheduler/scaler state |
| Slow step | Padding/length, dataloader, checkpoint recompute, kernel/device transfer |

Effective global examples/update = per-device microbatch × accumulation × data-parallel replicas;
effective tokens/update also depends on non-padding lengths. If accumulation windows contain different
token counts, dividing every microbatch equally is not identical to a true token-weighted large batch.
Decide and implement the desired normalization.

Clip global norm after unscaling. Log pre-clip norm and proportion clipped. Exclude norm/bias from
weight decay according to an explicit parameter-group rule. Validate optimizer/scheduler step counts
against accumulation; an off-by-K schedule silently changes training.


## Exercises

    1. Add a validation loop under `torch.no_grad()` and early stopping.
2. Compare decoupled weight decay with L2 regularization in Adam.
3. Save a checkpoint at step 15 and verify resumed training matches a continuous run.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
